# **HANDS - ON: INTEGRACIÓN ENTRE LLAMA Y WHATSAPP**

Una vez vista la masterclass ***Integración entre Llama y WhatsApp***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1ks58AJ1dS_aeNw_Q4pnYR05Dokr0Dwi6?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [19]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')
WABA_ID = userdata.get('WABA_TOKEN_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [2]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 102 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (2,546 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/l

In [3]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servidor levante

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [4]:
# Instalar dependencias
!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 8.4 MB/s eta 0:00:00
  

In [5]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [6]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import json

app = FastAPI()

mensajes_recibidos = []
ids_procesados = set()

In [7]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que asocia cada número de teléfono con su `conversation_id` de **Llama Stack** — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación.

In [8]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **DEFINIR LA HERRAMIENTA: CONSULTAR DISPONIBILIDAD Y PREGUNTAS FRECUENTES**

Hasta el Tema anterior, Llama solo generaba texto a partir de texto — si un usuario preguntaba por un horario, el modelo podía inventar una respuesta que sonara razonable, sin ninguna garantía de que fuera real. **Function calling** resuelve esto: se le describe a Llama una función real de tu código (nombre, qué hace, qué parámetros necesita), y el modelo, cuando lo considera necesario, pide que se ejecute esa función en vez de responder directamente.

Aquí se definen dos herramientas: `consultar_disponibilidad`, que revisa horarios reales del taller, y `buscar_pregunta_frecuente`, que busca la respuesta a dudas comunes (horarios, garantía, formas de pago) en una base de preguntas frecuentes real — una alternativa más ligera a montar un pipeline de RAG completo (embeddings + recuperación) aparte: en vez de eso, la búsqueda se expone como una función más que Llama puede pedir ejecutar, bajo el mismo mecanismo de function calling.

In [9]:
# Definir la funcion real y su descripcion para Llama: consultar disponibilidad

import re
from datetime import datetime

DISPONIBILIDAD = {
    "2026-10-06": ["09:00", "11:00", "15:00"],
    "2026-11-07": ["10:00", "13:00"],
}

MESES = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
}

def normalizar_fecha(fecha_texto):
    fecha_texto = fecha_texto.strip().lower()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", fecha_texto):
        _, mes, dia = fecha_texto.split("-")
        return f"{datetime.now().year:04d}-{mes}-{dia}"

    match = re.search(r"(\d{1,2})\s+de\s+([a-zñ]+)(?:\s+de\s+(\d{4}))?", fecha_texto)
    if match:
        dia, mes_texto, anio_texto = match.groups()
        mes = MESES.get(mes_texto)
        if mes:
            anio = int(anio_texto) if anio_texto else datetime.now().year
            return f"{anio:04d}-{mes:02d}-{int(dia):02d}"

    return fecha_texto

def consultar_disponibilidad(fecha=None):
    if not fecha:
        dias = ", ".join(DISPONIBILIDAD.keys())
        return f"Los dias con horarios disponibles son: {dias}. Dime cual te interesa para darte los horarios exactos."

    fecha = normalizar_fecha(fecha)
    horarios = DISPONIBILIDAD.get(fecha, [])
    if not horarios:
        return f"No hay horarios disponibles para el {fecha}."
    return f"Horarios disponibles para el {fecha}: {', '.join(horarios)}."

HERRAMIENTA_DISPONIBILIDAD = {
    "type": "function",
    "name": "consultar_disponibilidad",
    "description": "Consulta los horarios disponibles para llevar el auto al taller. Si el usuario da una fecha especifica, se le devuelven los horarios de ese dia; si pregunta de forma general (sin fecha), se le devuelve la lista de dias con disponibilidad.",
    "parameters": {
        "type": "object",
        "properties": {
            "fecha": {
                "type": "string",
                "description": "Fecha a consultar, en formato YYYY-MM-DD. Si el usuario no menciona el año, asume que es el año actual. Omite este parametro si el usuario no dio una fecha especifica."
            }
        },
        "required": []
    }
}

In [10]:
# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente

import unicodedata

def quitar_acentos(texto):
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

PREGUNTAS_FRECUENTES = {
    "horario": "El taller abre de lunes a sabado, de 8:00 a 18:00.",
    "garantia": "Todas las reparaciones tienen 90 dias de garantia en mano de obra.",   # raiz, no palabra completa
    "pago": "Aceptamos efectivo, tarjeta y transferencia.",
    "cotizacion": "La cotizacion es gratuita, solo se cobra si decides hacer la reparacion.",  # raiz
}

def buscar_pregunta_frecuente(pregunta):
    pregunta_norm = quitar_acentos(pregunta.lower())
    for raiz, respuesta in PREGUNTAS_FRECUENTES.items():
        if raiz in pregunta_norm:
            return respuesta
    return "No tenemos una respuesta guardada para esa pregunta; un miembro del equipo puede confirmarte directamente."

HERRAMIENTA_FAQ = {
    "type": "function",
    "name": "buscar_pregunta_frecuente",
    "description": "Busca la respuesta a una pregunta frecuente del taller (horarios, garantia, formas de pago, cotizaciones) en una base de preguntas frecuentes real.",
    "parameters": {
        "type": "object",
        "properties": {
            "pregunta": {
                "type": "string",
                "description": "La pregunta del usuario, en texto libre."
            }
        },
        "required": ["pregunta"]
    }
}

HERRAMIENTAS = [HERRAMIENTA_DISPONIBILIDAD, HERRAMIENTA_FAQ]
FUNCIONES_DISPONIBLES = {
    "consultar_disponibilidad": consultar_disponibilidad,
    "buscar_pregunta_frecuente": buscar_pregunta_frecuente,
}

#### **GENERAR LA RESPUESTA CON HISTORIAL Y FUNCTION CALLING**

Se reenvía el historial reciente a Llama junto con las dos herramientas disponibles. Si el modelo pide ejecutar alguna (`respuesta.output` trae uno o más elementos con `type == "function_call"`), el código la ejecuta de verdad con los argumentos que Llama propuso, le devuelve el resultado en una segunda llamada, y solo entonces se genera la respuesta final en lenguaje natural. Las reglas fijas (urgencia y escalamiento) se resuelven antes de llegar aquí — ver la sección de endpoints.

In [11]:
# Definir la funcion de generacion de respuesta con memoria y function calling

SYSTEM_PROMPT = (
    "Eres el asistente de un taller mecanico. "
    "Responde de forma breve (2-3 lineas). "
    "Solo usa una herramienta cuando el usuario pregunte algo especifico que la herramienta pueda responder "
    "(disponibilidad de una fecha, o una pregunta frecuente concreta). No uses ninguna herramienta para "
    "confirmaciones o respuestas cortas del usuario (como 'si, por favor' o una hora). "
    "No puedes agendar, confirmar ni programar citas de ninguna forma — solo puedes consultar disponibilidad. "
    "Si el usuario quiere agendar una cita, dile que un miembro del equipo se pondra en contacto para confirmarla. "
    "Si una herramienta no encuentra una respuesta, dilo honestamente en vez de inventar un dato. "
    "Si una herramienta SI regresa un resultado real, siempre comunicalo claramente al usuario; nunca lo ignores ni te niegues a compartirlo. "
    "Responde siempre de forma directa con la informacion final, basandote unicamente en el resultado real de las "
    "herramientas; nunca inventes datos que no vengan de una herramienta. "
    "Nunca describas tu proceso, ni menciones nombres de funciones, parametros o claves de un resultado."
)

def generar_respuesta(numero, texto_usuario):
    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
        tools=HERRAMIENTAS,
    )

    llamadas_funcion = [item for item in respuesta.output if item.type == "function_call"]
    if llamadas_funcion:
        resultados = []
        for llamada in llamadas_funcion:
            funcion = FUNCIONES_DISPONIBLES[llamada.name]
            argumentos = json.loads(llamada.arguments)
            resultado = funcion(**argumentos)
            resultados.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": resultado,
            })
        respuesta = client.responses.create(
            model="ollama/llama3.1:8b",
            input=resultados,
            conversation=conversation_id,
            instructions=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
        )

    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [12]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params
    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))
    return {"error": "token invalido"}

In [13]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)
    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        wamid = mensaje["id"]
        if wamid in ids_procesados:
            return {"status": "ok"}  # Meta reenvio el mismo mensaje; ya se proceso, se ignora
        ids_procesados.add(wamid)
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass
    return {"status": "ok"}

In [14]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [15]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

In [16]:
# Definir funcion de procesamiento: reglas fijas de urgencia, escalamiento y saludo, o respuesta de Llama

import requests
from llama_stack_client import APITimeoutError, APIConnectionError, APIStatusError

FRASES_URGENCIA = ["me quede varado", "no arranca", "choque", "tuve un accidente", "emergencia"]
FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "asistente humano"]
FRASES_SALUDO = ["hola", "buenas", "buenos dias", "buenos días", "buenas tardes", "buenas noches", "que tal", "buen dia", "buen día"]

def es_solo_saludo(texto_low):
    texto_limpio = texto_low.strip(" ,.!¡¿?")
    return texto_limpio in FRASES_SALUDO

def enviar_aviso_de_error(numero):
    try:
        enviar_respuesta(numero, "Estamos teniendo un problema tecnico. Intenta de nuevo en un momento.")
    except Exception:
        pass  # si ni el aviso de error se pudo mandar, no hay mas que intentar aqui

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_URGENCIA):
                enviar_respuesta(numero, "Eso suena urgente: comunicate directamente a nuestra linea de asistencia en carretera.")
            elif any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            elif es_solo_saludo(texto_low):
                enviar_respuesta(numero, "Hola, soy el asistente del taller mecanico. ¿En que puedo ayudarte?")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)

        except APITimeoutError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "llama_timeout",
                "detalle": "Llama Stack conecto pero la generacion tardo demasiado.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

        except APIConnectionError as e:
            llama_stack_vivo = proceso_llama_stack.poll() is None
            ollama_vivo = proceso_ollama.poll() is None
            if not llama_stack_vivo:
                causa, detalle = "llama_stack_caido", f"El proceso de Llama Stack ya no esta corriendo (Ollama vivo: {ollama_vivo})."
            elif not ollama_vivo:
                causa, detalle = "ollama_caido", "El proceso de Ollama ya no esta corriendo."
            else:
                causa, detalle = "conexion_desconocida", "No se pudo conectar a Llama Stack aunque ambos procesos parecen vivos."
            errores_procesamiento.append({"numero": numero, "texto": texto, "causa": causa, "detalle": detalle, "error": repr(e)})
            enviar_aviso_de_error(numero)

        except APIStatusError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "llama_stack_error_http",
                "detalle": f"Llama Stack respondio con error HTTP {e.status_code}.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

        except KeyError as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "herramienta_no_reconocida",
                "detalle": f"El modelo pidio ejecutar una herramienta que no existe: {e}.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

        except requests.exceptions.RequestException as e:
            # Este es el unico que NO intenta avisar al usuario: si fallo mandar el mensaje
            # a WhatsApp, intentar mandar el aviso de error fallaria por la misma razon.
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "whatsapp_api_error",
                "detalle": "Fallo el envio a la API de WhatsApp (graph.facebook.com).",
                "error": repr(e),
            })

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero, "texto": texto, "causa": "desconocido",
                "detalle": "Error no clasificado.",
                "error": repr(e),
            })
            enviar_aviso_de_error(numero)

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [18]:
!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000)
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(tunel_publico.public_url + "/webhook")

def correr_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

URL a configurar como Callback URL en el dashboard de Meta:
https://8164-104-196-53-205.ngrok-free.app/webhook


In [20]:
# Ver suscripciones de apps en WABA

r = requests.get(
    f"https://graph.facebook.com/v21.0/{WABA_ID}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)

print("Suscripciones actuales:", r.json())

Suscripciones actuales: {'data': [{'whatsapp_business_api_data': {'category': 'Negocios', 'link': 'https://www.facebook.com/games/?app_id=2202427980234937', 'name': 'WA DevX Webhook Events 1P App', 'id': '2202427980234937'}}]}


In [21]:
# Suscribir la app al WABA

r2 = requests.post(
    f"https://graph.facebook.com/v21.0/{WABA_ID}/subscribed_apps",
    headers={"Authorization": f"Bearer {WHATSAPP_TOKEN}"}
)

print("Resultado de suscripción:", r2.json())

Resultado de suscripción: {'success': True}


In [ ]:
# Escenario 1: Mandar mensajes al bot por disponibilidad general

# Mensaje 1:
# Buenas tardes.

# Mensaje 2:
# Necesito llevar mi carro al taller, ¿qué días tienen espacio?

# Mensaje 3:
# Perfecto, ¿y a qué hora el 6 de octubre?

In [22]:
import requests
import time
import uuid

def simular_mensaje_whatsapp(texto, numero_origen="5215512345678"):
    # Simulamos el formato JSON que envía el Webhook de Meta
    wamid = f"wamid.{uuid.uuid4().hex}"
    payload = {
        "object": "whatsapp_business_account",
        "entry": [
            {
                "id": "1149500037741062",
                "changes": [
                    {
                        "value": {
                            "messaging_product": "whatsapp",
                            "metadata": {
                                "display_phone_number": "15550000000",
                                "phone_number_id": "1408886008964243"
                            },
                            "contacts": [
                                {
                                    "profile": {
                                        "name": "Cliente de Prueba"
                                    },
                                    "wa_id": numero_origen
                                }
                            ],
                            "messages": [
                                {
                                    "from": numero_origen,
                                    "id": wamid,
                                    "timestamp": str(int(time.time())),
                                    "text": {
                                        "body": texto
                                    },
                                    "type": "text"
                                }
                            ]
                        },
                        "field": "messages"
                    }
                ]
            }
        ]
    }

    response = requests.post("http://localhost:8000/webhook", json=payload)
    print(f"Enviado: '{texto}' | Respuesta del webhook: {response.status_code} {response.json()}")

# --- Simulación del Escenario 1 ---
print("Simulando Escenario 1...")

# Mensaje 1
simular_mensaje_whatsapp("Buenas tardes.")
time.sleep(3)

# Mensaje 2
simular_mensaje_whatsapp("Necesito llevar mi carro al taller, ¿qué días tienen espacio?")
time.sleep(8) # Damos tiempo a Llama de procesar y llamar herramientas

# Mensaje 3
simular_mensaje_whatsapp("Perfecto, ¿y a qué hora el 6 de octubre?")
time.sleep(8)

print("\n--- Historial de respuestas enviadas simuladas ---")
for r in respuestas_enviadas[-3:]:
    print(f"Hacia: {r['numero']} | Status WhatsApp: {r['status']} | Texto enviado: {r['texto']}")

Simulando Escenario 1...
INFO:     127.0.0.1:32930 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Buenas tardes.' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     127.0.0.1:32932 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Necesito llevar mi carro al taller, ¿qué días tienen espacio?' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     127.0.0.1:45630 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Perfecto, ¿y a qué hora el 6 de octubre?' | Respuesta del webhook: 200 {'status': 'ok'}

--- Historial de respuestas enviadas simuladas ---
Hacia: 525512345678 | Status WhatsApp: 400 | Texto enviado: {"error":{"message":"(#131030) Recipient phone number not in allowed list","code":131030,"type":"OAuthException","error_data":{"messaging_product":"whatsapp","details":"Número de teléfono del destinatario no incluido en la lista de autorizados: Añade el número de teléfono del destinatario a la lista de destinatarios y vuelve a intentarlo."},"fbtrace_id":"AoUd0qC3vqMRjRFbaW1rlvk"}}


In [23]:
# --- Simulación del Escenario 1 usando tu número de teléfono real ---
print("Simulando Escenario 1 con el número: 5212462168664...")

# Mensaje 1
simular_mensaje_whatsapp("Buenas tardes.", numero_origen="5212462168664")
time.sleep(3)

# Mensaje 2
simular_mensaje_whatsapp("Necesito llevar mi carro al taller, ¿qué días tienen espacio?", numero_origen="5212462168664")
time.sleep(8)

# Mensaje 3
simular_mensaje_whatsapp("Perfecto, ¿y a qué hora el 6 de octubre?", numero_origen="5212462168664")
time.sleep(8)

print("\n--- Historial de respuestas enviadas a tu número ---")
for r in respuestas_enviadas:
    if "2462168664" in r["numero"]:
        print(f"Hacia: {r['numero']} | Status WhatsApp: {r['status']} | Texto enviado: {r['texto']}")

Simulando Escenario 1 con el número: 5212462168664...
INFO:     127.0.0.1:60722 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Buenas tardes.' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     127.0.0.1:41286 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Necesito llevar mi carro al taller, ¿qué días tienen espacio?' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     2a03:2880:12ff:41:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:10ff:c:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:10ff:1:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     127.0.0.1:41310 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Perfecto, ¿y a qué hora el 6 de octubre?' | Respuesta del webhook: 200 {'status': 'ok'}

--- Historial de respuestas enviadas a tu número ---
Hacia: 522462168664 | Status WhatsApp: 200 | Texto enviado: {"messaging_product":"whatsapp","contacts":[{"input":"522462168664","wa_id":"5212462168664"}],"messages":[{"id":"wamid.HBgNNTIxMjQ2MjE2ODY2NBUCABEYEkQ3NDQ2NzRFNTc1NDlB

### **PROBAR EL FLUJO COMPLETO**

Se manda un mensaje preguntando por horarios disponibles en una fecha específica, para confirmar que la respuesta viene de `consultar_disponibilidad` y no de una invención de Llama; y un segundo mensaje pidiendo hablar con una persona, para confirmar que la regla fija de escalamiento sigue funcionando.

In [24]:
# --- Simulación del Escenario 3 usando tu número de teléfono real ---
print("Simulando Escenario 3 con el número: 5212462168664...")

# Mensaje 1
simular_mensaje_whatsapp("Buenas, ¿cuánto cuesta una cotización?", numero_origen="5212462168664")
time.sleep(8)

# Mensaje 2
simular_mensaje_whatsapp("¿Y qué métodos de pago aceptan?", numero_origen="5212462168664")
time.sleep(8)

# Mensaje 3
simular_mensaje_whatsapp("Perfecto, entonces agéndame para el 7 de octubre a las 10.", numero_origen="5212462168664")
time.sleep(8)

print("\n--- Historial de respuestas enviadas del Escenario 3 ---")
for r in respuestas_enviadas[-3:]:
    print(f"Hacia: {r['numero']} | Status WhatsApp: {r['status']} | Texto enviado: {r['texto']}")

Simulando Escenario 3 con el número: 5212462168664...
INFO:     127.0.0.1:59730 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Buenas, ¿cuánto cuesta una cotización?' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     2a03:2880:10ff:7:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:21ff:44:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:2ff:4b:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     127.0.0.1:47640 - "POST /webhook HTTP/1.1" 200 OK
Enviado: '¿Y qué métodos de pago aceptan?' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     2a03:2880:10ff:5c:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:2ff:4c:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:2ff:3:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     127.0.0.1:51408 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Perfecto, entonces agéndame para el 7 de octubre a las 10.' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     2a03:2880:22ff:48:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:   

In [ ]:
# Escenario 4: Mandar mensajes al bot por temas fuera de alcance

# Mensaje 1:
# Hola, ¿tienen servicio de grúa?

# Mensaje 2:
# ¿También atienden motos, o solo carros?

In [25]:
# --- Simulación del Escenario 4 usando tu número de teléfono real ---
print("Simulando Escenario 4 con el número: 5212462168664...")

# Mensaje 1
simular_mensaje_whatsapp("Hola, ¿tienen servicio de grúa?", numero_origen="5212462168664")
time.sleep(8)

# Mensaje 2
simular_mensaje_whatsapp("¿También atienden motos, o solo carros?", numero_origen="5212462168664")
time.sleep(8)

print("\n--- Historial de respuestas enviadas del Escenario 4 ---")
for r in respuestas_enviadas[-2:]:
    print(f"Hacia: {r['numero']} | Status WhatsApp: {r['status']} | Texto enviado: {r['texto']}")

Simulando Escenario 4 con el número: 5212462168664...
INFO:     127.0.0.1:58924 - "POST /webhook HTTP/1.1" 200 OK
Enviado: 'Hola, ¿tienen servicio de grúa?' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     2a03:2880:22ff:49:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:22ff:50:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:2ff:49:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     127.0.0.1:36350 - "POST /webhook HTTP/1.1" 200 OK
Enviado: '¿También atienden motos, o solo carros?' | Respuesta del webhook: 200 {'status': 'ok'}
INFO:     2a03:2880:10ff:4:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:10ff:71:::0 - "POST /webhook HTTP/1.1" 200 OK
INFO:     2a03:2880:2ff:73:::0 - "POST /webhook HTTP/1.1" 200 OK

--- Historial de respuestas enviadas del Escenario 4 ---
Hacia: 522462168664 | Status WhatsApp: 200 | Texto enviado: {"messaging_product":"whatsapp","contacts":[{"input":"522462168664","wa_id":"5212462168664"}],"messages":[{"id":"wamid.HBgNNTIxMj

In [26]:
# Visualizar resultados
print(f'Conversaciones: {conversaciones_por_numero}\n')
print(f'Mensajes recibidos: {mensajes_recibidos}\n')
print(f'Respuestas enviadas: {respuestas_enviadas}\n')
print(f'Errores procesados: {errores_procesamiento}\n')

Conversaciones: {'525512345678': 'conv_8cd3dc91b1a193a3bd7a0a1efe80e4e401566c87b10cf610', '522462168664': 'conv_c7fdaa69148dca1108cef9cb8a862181723fa2f47b4b4fd4'}

Mensajes recibidos: [{'object': 'whatsapp_business_account', 'entry': [{'id': '1149500037741062', 'changes': [{'value': {'messaging_product': 'whatsapp', 'metadata': {'display_phone_number': '15550000000', 'phone_number_id': '1408886008964243'}, 'contacts': [{'profile': {'name': 'Cliente de Prueba'}, 'wa_id': '5215512345678'}], 'messages': [{'from': '5215512345678', 'id': 'wamid.56266d52d0cd44e1a5bb026d3f38f30a', 'timestamp': '1792473349', 'text': {'body': 'Buenas tardes.'}, 'type': 'text'}]}, 'field': 'messages'}]}]}, {'object': 'whatsapp_business_account', 'entry': [{'id': '1149500037741062', 'changes': [{'value': {'messaging_product': 'whatsapp', 'metadata': {'display_phone_number': '15550000000', 'phone_number_id': '1408886008964243'}, 'contacts': [{'profile': {'name': 'Cliente de Prueba'}, 'wa_id': '5215512345678'}], 'm

In [27]:
# Verificar almacenamiento en Llama Stack

numero_a_revisar = '5212462168664'

if numero_a_revisar not in conversaciones_por_numero:
    print("No hay mensajes almacenados en estos momentos.")
else:
    conv_id = conversaciones_por_numero[numero_a_revisar]
    items = client.conversations.items.list(conversation_id=conv_id)

    for item in items.data:
        if item.type == "message":
            print(item.role, "->", item.content)
        elif item.type == "function_call":
            print("function_call ->", item.name, item.arguments)
        elif item.type == "function_call_output":
            print("function_call_output ->", item.output)
        else:
            print(item.type, "->", item)

No hay mensajes almacenados en estos momentos.


In [28]:
# Reiniciar el almacenamiento en Llama Stack

proceso_llama_stack.terminate()
import time; time.sleep(3)

!rm -rf ~/.llama/distributions/starter

proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321", timeout=300)
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


In [29]:
# Limpiar los registros en memoria, para una prueba nueva sin ruido de mensajes anteriores

mensajes_recibidos = []
respuestas_enviadas = []
errores_procesamiento = []
conversaciones_por_numero = {}

In [30]:
# Finalizar proceso de tunelizacion

ngrok.kill()
print('Proceso de ngrok cerrado.')

Proceso de ngrok cerrado.


### **DIAGNÓSTICO Y REINICIO**

Si el bot responde con datos que no corresponden al taller (fechas, textos u otro negocio), antes de reiniciar todo el entorno revisa primero si simplemente falta volver a correr alguna celda de arriba.

In [ ]:
# Reiniciar el almacenamiento en Llama Stack


In [ ]:
# Limpiar los registros en memoria, para una prueba nueva sin ruido de mensajes anteriores


In [ ]:
# Finalizar proceso de tunelizacion


## **CHALLENGE: AGENTE DE ATENCIÓN A PEDIDOS**

Una vez visto el ***Hands-On: Integración Llama + WhatsApp***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para la atención a pedidos de una tienda en línea: la misma memoria de conversación en Llama Stack, las mismas reglas fijas (urgencia, escalamiento y saludo) y las mismas dos herramientas de function calling, ahora como `consultar_pedido` y `buscar_pregunta_frecuente`.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [44]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 102 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [45]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servidor levante

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


Ollama listo.


2. Construye la base de conocimiento (RAG): divide las políticas de la tienda en fragmentos, conviértelos en embeddings con Ollama y define la búsqueda de los fragmentos más cercanos a una pregunta. Ajusta el UMBRAL con la celda de prueba.

In [46]:
# Definir la base de conocimiento y generar sus embeddings con Ollama

import math
import ollama

MODELO_LLM = "llama3.1:8b"
MODELO_EMBEDDINGS = "bge-m3"

documentos = [
    "Envíos: los pedidos se entregan en 3 a 5 días hábiles. El envío es gratis en compras mayores a $500; en compras menores cuesta $60.",
    "Seguimiento: al enviarse el pedido se comparte un número de guía, y el estado se puede consultar con el número de pedido.",
    "Devoluciones: se aceptan dentro de los 15 días posteriores a la entrega, con el producto sin usar y en su empaque original. El reembolso tarda de 5 a 7 días hábiles.",
    "Cambios: se puede cambiar talla o color dentro de los 15 días posteriores a la entrega, sujeto a disponibilidad.",
    "Cancelaciones: un pedido solo se puede cancelar mientras su estado sea 'preparando envío'. Una vez enviado ya no es posible cancelarlo.",
    "Formas de pago: se aceptan tarjeta de crédito o débito, transferencia y pago en tiendas de conveniencia.",
    "Facturación: la factura se solicita dentro del mismo mes de la compra, con los datos fiscales de la persona.",
    "Garantía: los productos electrónicos tienen 6 meses de garantía por defectos de fabricación.",
    "Horario de atención: de lunes a viernes de 9:00 a 18:00.",
    "Cambio de dirección: la dirección de entrega solo se puede modificar antes de que el pedido sea enviado.",
]

def generar_embeddings(textos):
    # Asegurar que el modelo de embeddings esté disponible
    try:
        r = ollama.embed(model=MODELO_EMBEDDINGS, input=textos)
    except ollama.ResponseError as e:
        if "not found" in str(e).lower():
            print(f"Descargando el modelo de embeddings {MODELO_EMBEDDINGS}...")
            ollama.pull(MODELO_EMBEDDINGS)
            r = ollama.embed(model=MODELO_EMBEDDINGS, input=textos)
        else:
            raise e
    return r["embeddings"]

embeddings_documentos = generar_embeddings(documentos)
print("Embeddings generados:", len(embeddings_documentos), "x", len(embeddings_documentos[0]))

Embeddings generados: 10 x 1024


In [47]:
# Definir la busqueda de fragmentos relevantes (la parte de recuperacion del RAG)

UMBRAL = 0.50  # similitud minima para considerar que la base de conocimiento cubre la pregunta

def similitud_coseno(a, b):
    producto = sum(x * y for x, y in zip(a, b))
    norma_a = math.sqrt(sum(x * x for x in a))
    norma_b = math.sqrt(sum(y * y for y in b))
    return producto / (norma_a * norma_b)

def similitudes(consulta):
    embedding_consulta = generar_embeddings([consulta])[0]
    return [similitud_coseno(doc, embedding_consulta) for doc in embeddings_documentos]

def recuperar_fragmentos(consulta, k=2):
    puntajes = similitudes(consulta)
    indices = sorted(range(len(puntajes)), key=lambda i: puntajes[i], reverse=True)[:k]
    return [documentos[i] for i in indices if puntajes[i] >= UMBRAL]

In [48]:
# Probar la recuperacion: revisar que fragmentos regresa y con que similitud (sirve para ajustar el UMBRAL)

for pregunta in ['¿Cuánto tarda el envío?', '¿Puedo cancelar un pedido?', '¿Venden zapatos deportivos?']:
    print(f"{pregunta} -> similitud maxima: {max(similitudes(pregunta)):.2f}")
    for fragmento in recuperar_fragmentos(pregunta):
        print("   ", fragmento[:80], "...")
    print()

¿Cuánto tarda el envío? -> similitud maxima: 0.74
    Envíos: los pedidos se entregan en 3 a 5 días hábiles. El envío es gratis en com ...
    Devoluciones: se aceptan dentro de los 15 días posteriores a la entrega, con el  ...

¿Puedo cancelar un pedido? -> similitud maxima: 0.76
    Cancelaciones: un pedido solo se puede cancelar mientras su estado sea 'preparan ...
    Cambio de dirección: la dirección de entrega solo se puede modificar antes de qu ...

¿Venden zapatos deportivos? -> similitud maxima: 0.40



3. Define las dos herramientas de function calling:

consultar_pedido(numero_pedido): una función real que consulta el estado de un pedido. Si el usuario no da un número de pedido, la función debe pedírselo en vez de inventar uno.
buscar_en_politicas(pregunta): la búsqueda RAG del paso anterior, expuesta como herramienta para que Llama decida cuándo consultarla.
Una función ejecutar_herramienta que ejecuta la que Llama pida y regresa un mensaje claro si algo falla.

In [49]:
# Definir la funcion real y su descripcion para Llama: consultar pedido

import re

PEDIDOS = {
    "4521": {"estado": "en camino", "fecha_estimada": "2026-10-05"},
    "4522": {"estado": "entregado", "fecha_estimada": "2026-10-01"},
    "4523": {"estado": "preparando envio", "fecha_estimada": "2026-10-08"},
}

def normalizar_numero_pedido(texto):
    # el modelo a veces manda "#4521" o "pedido 4521": se queda solo con los digitos
    digitos = re.findall(r"\d+", str(texto))
    return digitos[0] if digitos else ""

def consultar_pedido(numero_pedido=None):
    numero = normalizar_numero_pedido(numero_pedido) if numero_pedido else ""
    if not numero:
        return "Para consultar un pedido necesito su numero de pedido. ¿Me lo puedes compartir?"

    pedido = PEDIDOS.get(numero)
    if not pedido:
        return f"No encontramos un pedido con el numero {numero}."
    return f"El pedido {numero} esta {pedido['estado']}, con fecha estimada {pedido['fecha_estimada']}."

HERRAMIENTA_PEDIDO = {
    "type": "function",
    "function": {
        "name": "consultar_pedido",
        "description": "Consulta el estado real de un pedido de la tienda a partir de su numero. Si el usuario no dio un numero de pedido, omite el parametro.",
        "parameters": {
            "type": "object",
            "properties": {
                "numero_pedido": {
                    "type": "string",
                    "description": "Numero del pedido a consultar, solo digitos. Omite este parametro si el usuario no dio un numero de pedido.",
                }
            },
            "required": [],
        },
    },
}

In [50]:
# Definir la herramienta de busqueda en politicas (RAG) y el ejecutor de herramientas

def buscar_en_politicas(pregunta):
    fragmentos = recuperar_fragmentos(pregunta)
    if not fragmentos:
        return "Las politicas de la tienda no cubren esa pregunta; un miembro del equipo puede confirmarte directamente."
    return "Politicas de la tienda: " + " | ".join(fragmentos)

HERRAMIENTA_POLITICAS = {
    "type": "function",
    "function": {
        "name": "buscar_en_politicas",
        "description": "Busca en las politicas de la tienda la respuesta a dudas sobre envios, devoluciones, cambios, cancelaciones, formas de pago, facturacion, garantia y horario de atencion.",
        "parameters": {
            "type": "object",
            "properties": {
                "pregunta": {"type": "string", "description": "La pregunta del usuario sobre las politicas, en texto libre."}
            },
            "required": ["pregunta"],
        },
    },
}

HERRAMIENTAS = [HERRAMIENTA_PEDIDO, HERRAMIENTA_POLITICAS]
FUNCIONES_DISPONIBLES = {
    "consultar_pedido": consultar_pedido,
    "buscar_en_politicas": buscar_en_politicas,
}

def ejecutar_herramienta(nombre, argumentos):
    print(f"   [herramienta] {nombre}({dict(argumentos)})")
    funcion = FUNCIONES_DISPONIBLES.get(nombre)
    if funcion is None:
        return f"La herramienta {nombre} no existe."
    try:
        return funcion(**argumentos)
    except TypeError:
        return f"No se pudo ejecutar {nombre}: argumentos invalidos."

4. Construye la memoria de conversación y genera la respuesta con function calling:

Guarda el historial de cada número de teléfono por separado.
Manda el historial a Llama junto con las dos herramientas. Si Llama pide ejecutar alguna, ejecútala, devuelve el resultado al modelo y vuelve a llamarlo para obtener la respuesta final.

In [51]:
# Memoria de conversacion por numero de telefono

historial_por_numero = {}

def obtener_historial(numero):
    if numero not in historial_por_numero:
        historial_por_numero[numero] = []
    return historial_por_numero[numero]

In [52]:
# Definir la generacion de respuesta con historial y function calling

SYSTEM_PROMPT = (
    "Eres el asistente de atencion a clientes de una tienda en linea. "
    "Responde de forma breve (2-3 lineas). "
    "Solo usa una herramienta cuando el usuario pregunte algo especifico que la herramienta pueda responder "
    "(el estado de un pedido, o una duda sobre las politicas de la tienda). No uses ninguna herramienta para "
    "confirmaciones o respuestas cortas del usuario (como 'si, por favor' o 'gracias'). "
    "No puedes cancelar, modificar ni reembolsar pedidos de ninguna forma — solo puedes consultar su estado y explicar las politicas. "
    "Si el usuario quiere cancelar, modificar o reembolsar un pedido, dile que un miembro del equipo se pondra en contacto. "
    "Si una herramienta no encuentra una respuesta, dilo honestamente en vez de inventar un dato. "
    "Si una herramienta SI regresa un resultado real, siempre comunicalo claramente al usuario; nunca lo ignores ni te niegues a compartirlo. "
    "Responde siempre de forma directa con la informacion final, basandote unicamente en el resultado real de las "
    "herramientas; nunca inventes datos que no vengan de una herramienta. "
    "Nunca describas tu proceso, ni menciones nombres de funciones, parametros o claves de un resultado."
)

def generar_respuesta(numero, texto_usuario):
    historial = obtener_historial(numero)
    mensajes = [{"role": "system", "content": SYSTEM_PROMPT}] + historial + [{"role": "user", "content": texto_usuario}]

    for _ in range(3):  # limite de rondas de herramientas, para evitar ciclos
        respuesta = ollama.chat(model=MODELO_LLM, messages=mensajes, tools=HERRAMIENTAS)
        llamadas = respuesta.message.tool_calls
        if not llamadas:
            break
        mensajes.append(respuesta.message)
        for llamada in llamadas:
            resultado = ejecutar_herramienta(llamada.function.name, llamada.function.arguments)
            mensajes.append({"role": "tool", "tool_name": llamada.function.name, "content": resultado})

    texto_respuesta = respuesta.message.content or "No pude completar la consulta. Un miembro del equipo puede ayudarte directamente."

    # En el historial solo se guardan los turnos de texto, no los mensajes internos de herramientas
    historial.append({"role": "user", "content": texto_usuario})
    historial.append({"role": "assistant", "content": texto_respuesta})
    return texto_respuesta

5. Agrega las reglas fijas:

Si el usuario menciona un problema con un cargo (ej. "me cobraron dos veces", "cargo no reconocido", "fraude"), se responde con un mensaje fijo de prioridad, sin pasar por Llama.
Si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
Si el mensaje es solo un saludo, se responde con un saludo fijo presentando al asistente.

In [53]:
# Definir las reglas fijas y la funcion que procesa cada mensaje

FRASES_URGENCIA = ["me cobraron doble", "me cobraron dos veces", "cargo no reconocido", "fraude", "emergencia"]
FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano", "asistente humano"]
FRASES_SALUDO = ["hola", "buenas", "buenos dias", "buenos días", "buenas tardes", "buenas noches", "que tal", "buen dia", "buen día"]

def es_solo_saludo(texto_low):
    return texto_low.strip(" ,.!¡¿?") in FRASES_SALUDO

def procesar_mensaje(numero, texto):
    texto_low = texto.lower()
    if any(frase in texto_low for frase in FRASES_URGENCIA):
        return "Eso suena urgente: comunicate directamente a nuestra linea de atencion para cargos y aclaraciones."
    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."
    if es_solo_saludo(texto_low):
        return "Hola, soy el asistente de atencion a pedidos de la tienda. ¿En que puedo ayudarte?"
    return generar_respuesta(numero, texto)

6. Expón el servidor con ngrok y prueba el flujo completo:

Manda un mensaje preguntando por el estado de un pedido y confirma que la respuesta viene de consultar_pedido (dato real), no de una invención de Llama.
Manda una pregunta frecuente y confirma que dispara buscar_pregunta_frecuente.
Manda un mensaje sobre un cargo duplicado y confirma que dispara la regla de urgencia, sin pasar por Llama.
Manda un mensaje pidiendo hablar con una persona y confirma que dispara la regla de escalamiento.

In [54]:
# Simular una conversacion completa (las lineas [herramienta] muestran que funcion pidio Llama)

numero = "5212462168664"
mensajes_prueba = [
    "Hola",
    "¿Cómo va mi pedido 4521?",
    "¿Y cuánto tarda el envío en general?",
    "¿Puedo cancelar el pedido 4523?",
    "¿Dónde está mi pedido?",
    "Es el 4599",
    "Me cobraron dos veces mi pedido",
    "Mejor quiero hablar con una persona",
]

for texto in mensajes_prueba:
    print(f"Usuario: {texto}")
    print(f"Bot: {procesar_mensaje(numero, texto)}\n")

Usuario: Hola
Bot: Hola, soy el asistente de atencion a pedidos de la tienda. ¿En que puedo ayudarte?

Usuario: ¿Cómo va mi pedido 4521?
   [herramienta] consultar_pedido({'numero_pedido': '4521'})
Bot: El pedido 4521 esta en camino, con fecha estimada 2026-10-05.

Usuario: ¿Y cuánto tarda el envío en general?
   [herramienta] buscar_en_politicas({'pregunta': '¿Cuánto tarda el envío en general?'})
Bot: El envío generalmente tarda entre 3 a 5 días hábiles.

Usuario: ¿Puedo cancelar el pedido 4523?
   [herramienta] buscar_en_politicas({'pregunta': '¿Puedo cancelar un pedido?'})
Bot: Lo siento, pero debido a que el estado de tu pedido 4523 ya no es 'preparando envío', no podemos cancelarlo. Un miembro de nuestro equipo se pondrá en contacto contigo para discutir opciones.

Usuario: ¿Dónde está mi pedido?
   [herramienta] consultar_pedido({'numero_pedido': '4521'})
Bot: Lo siento, pero no tengo información sobre el pedido 4523. Si necesitas ayuda, por favor, comunícate con nuestro equipo.


In [55]:
# Revisar el historial guardado

for m in obtener_historial(numero):
    print(f"{m['role']}: {m['content']}")

user: ¿Cómo va mi pedido 4521?
assistant: El pedido 4521 esta en camino, con fecha estimada 2026-10-05.
user: ¿Y cuánto tarda el envío en general?
assistant: El envío generalmente tarda entre 3 a 5 días hábiles.
user: ¿Puedo cancelar el pedido 4523?
assistant: Lo siento, pero debido a que el estado de tu pedido 4523 ya no es 'preparando envío', no podemos cancelarlo. Un miembro de nuestro equipo se pondrá en contacto contigo para discutir opciones.
user: ¿Dónde está mi pedido?
assistant: Lo siento, pero no tengo información sobre el pedido 4523. Si necesitas ayuda, por favor, comunícate con nuestro equipo.
user: Es el 4599
assistant: Lo siento, pero no tenemos información sobre un pedido con el número 4599. Si necesitas ayuda, por favor, comunícate con nuestro equipo.
